# HydroTwin Data Preparation

This notebook prepares the SCADA data for the HydroTwin predictive maintenance system.

The study focuses on two connected hydropower components:

1. Turbine Guide Bearing (TGB)
2. Upper Guide Bearing (UGB)

For each component, vibration, temperature, unit speed, and generator power are used to support vibration forecasting.

The prepared datasets will later be used with Linear Regression, LSTM, and XGBoost.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

## 1. Project Paths

The project folders are defined relative to the HydroTwin directory.

In [ ]:
PROJECT_ROOT = Path("..").resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"

PREPARED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw data folder:")
print(RAW_DIR)

print("\nPrepared data folder:")
print(PREPARED_DIR)

## 2. Check Raw SCADA Files

The available monthly SCADA files are checked before data loading.

In [ ]:
csv_files = sorted(
    RAW_DIR.glob("*.csv")
)

print("Number of CSV files:", len(csv_files))

for file in csv_files:
    print(file.name)

## 3. Inspect One Monthly Dataset

The June SCADA dataset is loaded first to inspect the available variables and confirm the measurements required for the Turbine Guide Bearing and Upper Guide Bearing.

In [ ]:
june_path = RAW_DIR / "June.csv"

june = pd.read_csv(
    june_path
)

print("June dataset shape:")
print(june.shape)

june.head()

In [ ]:
print("Number of columns:", len(june.columns))

for i, column in enumerate(
    june.columns,
    start=1
):
    print(i, column)

In [ ]:
bearing_columns = [
    column
    for column in june.columns
    if any(
        keyword in column.lower()
        for keyword in [
            "turbine",
            "tgb",
            "upper",
            "ugb",
            "vibr",
            "temperature",
            "speed",
            "generator p"
        ]
    )
]

print(
    "Possible relevant columns:",
    len(bearing_columns)
)

for column in bearing_columns:
    print(column)

In [ ]:
june.shape

In [ ]:
for i, column in enumerate(june.columns, start=1):
    print(i, column)

In [ ]:
for column in bearing_columns:
    print(column)

## 4. Select TGB and UGB Variables

Only the variables required for the Turbine Guide Bearing and Upper Guide Bearing models are retained.

The selected variables include timestamp, generator power, unit speed, component temperature, and horizontal vibration in the positive X direction.

In [ ]:
TIMESTAMP_COL = "Point Name"

POWER_COL = "G1 Outlet Of Generator P"

SPEED_COL = (
    "Neelum-Jhelum:AIN.G1 Unit Speed(GOV system)"
)

UGB_TEMP_COL = (
    "Neelum-Jhelum:AIN.G1 RIO1 upper guide pad 1# temperature"
)

TGB_TEMP_COL = (
    "Neelum-Jhelum:AIN.G1 RIO1 turbine guide pad 1# temperature"
)

UGB_VIB_COL = (
    "Neelum-Jhelum:AIN.G1 vibrarion runout system "
    "UGB horizontal vibrarion +X"
)

TGB_VIB_COL = (
    "Neelum-Jhelum:AIN.G1 vibrarion runout system "
    "TGB horizontal vibrarion +X"
)

In [ ]:
selected_columns = [
    TIMESTAMP_COL,
    POWER_COL,
    SPEED_COL,
    TGB_TEMP_COL,
    TGB_VIB_COL,
    UGB_TEMP_COL,
    UGB_VIB_COL
]

In [ ]:
june_selected = june[
    selected_columns
].copy()

print(
    "Selected dataset shape:",
    june_selected.shape
)

june_selected.head()

## 5. Rename Variables

The original SCADA tag names are renamed to shorter and consistent variable names to simplify later preprocessing and modelling.

In [ ]:
june_selected = june_selected.rename(
    columns={
        TIMESTAMP_COL: "timestamp",
        POWER_COL: "generator_power",
        SPEED_COL: "unit_speed",

        TGB_TEMP_COL: "tgb_temperature",
        TGB_VIB_COL: "tgb_vibration",

        UGB_TEMP_COL: "ugb_temperature",
        UGB_VIB_COL: "ugb_vibration"
    }
)

In [ ]:
june_selected.head()

In [ ]:
june_selected.info()

In [ ]:
june_selected.dtypes

## 6. Initial Data Quality Check

The selected variables are checked for missing values before further preprocessing.

In [ ]:
missing_values = (
    june_selected
    .isna()
    .sum()
)

missing_values

In [ ]:
june_selected.describe()

## 7. Timestamp Conversion

The timestamp variable is converted from text format into a datetime format to support chronological sorting and time series analysis.

In [ ]:
june_selected["timestamp"] = pd.to_datetime(
    june_selected["timestamp"],
    format="%Y/%m/%d %H:%M:%S",
    errors="coerce"
)

june_selected.dtypes

In [ ]:
print(
    "Invalid timestamps:",
    june_selected["timestamp"].isna().sum()
)

## 8. Inspect Missing Values

Rows containing missing sensor measurements are inspected before deciding how they should be handled.

In [ ]:
missing_rows = june_selected[
    june_selected.isna().any(axis=1)
]

missing_rows

In [ ]:
time_differences = (
    june_selected["timestamp"]
    .sort_values()
    .diff()
    .value_counts()
)

time_differences.head()

In [ ]:
duplicate_timestamps = (
    june_selected["timestamp"]
    .duplicated()
    .sum()
)

print(
    "Duplicate timestamps:",
    duplicate_timestamps
)

## 9. Initial Sensor Statistics

Descriptive statistics are reviewed to understand the range and distribution of the selected operating and bearing measurements.

In [ ]:
june_selected.describe()

## 10. Remove Incomplete Records

Records with missing values in the selected modelling variables are removed because complete measurements are required for consistent multivariate forecasting.

In [ ]:
june_selected = (
    june_selected
    .dropna()
    .reset_index(drop=True)
)

print(
    "June shape after removing incomplete rows:",
    june_selected.shape
)

print(
    "\nMissing values after cleaning:"
)

print(
    june_selected.isna().sum()
)

## 11. Load All Monthly SCADA Data

The same seven variables are extracted from each monthly SCADA dataset. The column names are standardized before the monthly datasets are combined.

In [ ]:
file_names = [
    "June.csv",
    "July.csv",
    "37771965_Aug.csv",
    "SEP.csv",
    "Oct.csv",
    "Nov.csv"
]

In [ ]:
monthly_data = []

In [ ]:
def load_month(file_name):

    file_path = RAW_DIR / file_name

    df = pd.read_csv(
        file_path
    )

    df = df[
        selected_columns
    ].copy()

    df = df.rename(
        columns={
            TIMESTAMP_COL: "timestamp",
            POWER_COL: "generator_power",
            SPEED_COL: "unit_speed",

            TGB_TEMP_COL: "tgb_temperature",
            TGB_VIB_COL: "tgb_vibration",

            UGB_TEMP_COL: "ugb_temperature",
            UGB_VIB_COL: "ugb_vibration"
        }
    )

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        errors="coerce"
    )

    df["source_file"] = file_name

    return df

In [ ]:
for file_name in file_names:

    month_df = load_month(
        file_name
    )

    monthly_data.append(
        month_df
    )

    print(
        file_name,
        "->",
        month_df.shape
    )

In [ ]:
combined_data = pd.concat(
    monthly_data,
    ignore_index=True
)

In [ ]:
print(
    "Combined dataset shape:",
    combined_data.shape
)

combined_data.head()

In [ ]:
combined_data.groupby(
    "source_file"
).size()

In [ ]:
combined_data.groupby(
    "source_file"
)[
    [
        "generator_power",
        "unit_speed",
        "tgb_temperature",
        "tgb_vibration",
        "ugb_temperature",
        "ugb_vibration"
    ]
].count()

In [ ]:
combined_data.isna().sum()

## 12. Inspect Incomplete Records

All records containing missing timestamps or sensor measurements are inspected before any rows are removed.

In [ ]:
incomplete_rows = combined_data[
    combined_data.isna().any(axis=1)
].copy()

print(
    "Number of incomplete rows:",
    len(incomplete_rows)
)

incomplete_rows

In [ ]:
for file_name in file_names:

    month_data = combined_data[
        combined_data["source_file"] == file_name
    ]

    print("\n", file_name)

    print(
        month_data.isna().sum()
    )

In [ ]:
valid_timestamps = combined_data[
    "timestamp"
].dropna()

duplicate_timestamps = (
    valid_timestamps
    .duplicated()
    .sum()
)

print(
    "Duplicate timestamps across all months:",
    duplicate_timestamps
)

In [ ]:
print(
    "Earliest timestamp:",
    combined_data["timestamp"].min()
)

print(
    "Latest timestamp:",
    combined_data["timestamp"].max()
)

In [ ]:
monthly_ranges = (
    combined_data
    .groupby("source_file")["timestamp"]
    .agg(
        ["min", "max"]
    )
)

monthly_ranges

## 13. Remove Incomplete Records

Five incomplete SCADA records are removed because the selected modelling variables require complete measurements for both guide bearing components.

In [ ]:
clean_data = (
    combined_data
    .dropna()
    .copy()
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print(
    "Shape before cleaning:",
    combined_data.shape
)

print(
    "Shape after cleaning:",
    clean_data.shape
)

print(
    "\nMissing values after cleaning:"
)

print(
    clean_data.isna().sum()
)

In [ ]:
print(
    "First timestamp:",
    clean_data["timestamp"].min()
)

print(
    "Last timestamp:",
    clean_data["timestamp"].max()
)

print(
    "Total cleaned observations:",
    len(clean_data)
)

## 14. Operating Condition Analysis

Generator power and unit speed are examined to distinguish normal operating conditions from startup, shutdown, and non operating periods.

In [ ]:
operating_summary = clean_data[
    [
        "generator_power",
        "unit_speed"
    ]
].describe(
    percentiles=[
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

operating_summary

In [ ]:
speed_checks = pd.DataFrame({
    "Condition": [
        "Speed <= 0",
        "Speed < 20",
        "Speed < 50",
        "Speed < 80",
        "Speed >= 80",
        "Speed >= 90"
    ],

    "Count": [
        (clean_data["unit_speed"] <= 0).sum(),
        (clean_data["unit_speed"] < 20).sum(),
        (clean_data["unit_speed"] < 50).sum(),
        (clean_data["unit_speed"] < 80).sum(),
        (clean_data["unit_speed"] >= 80).sum(),
        (clean_data["unit_speed"] >= 90).sum()
    ]
})

speed_checks

In [ ]:
power_checks = pd.DataFrame({
    "Condition": [
        "Power <= 0",
        "Power < 50",
        "Power < 100",
        "Power < 150",
        "Power >= 150",
        "Power >= 200"
    ],

    "Count": [
        (clean_data["generator_power"] <= 0).sum(),
        (clean_data["generator_power"] < 50).sum(),
        (clean_data["generator_power"] < 100).sum(),
        (clean_data["generator_power"] < 150).sum(),
        (clean_data["generator_power"] >= 150).sum(),
        (clean_data["generator_power"] >= 200).sum()
    ]
})

power_checks

In [ ]:
clean_data[
    [
        "timestamp",
        "generator_power",
        "unit_speed",
        "tgb_vibration",
        "ugb_vibration"
    ]
].sort_values(
    "unit_speed"
).head(20)

In [ ]:
clean_data[
    [
        "timestamp",
        "generator_power",
        "unit_speed",
        "tgb_vibration",
        "ugb_vibration"
    ]
].sort_values(
    "generator_power"
).head(20)

## 15. Operating Behaviour Visualization

Unit speed and generator power are visualized over time to identify periods of normal operation, startup, shutdown, and inactivity.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(
    figsize=(14, 5)
)

plt.plot(
    clean_data["timestamp"],
    clean_data["unit_speed"]
)

plt.xlabel(
    "Time"
)

plt.ylabel(
    "Unit Speed"
)

plt.title(
    "Unit Speed Over Time"
)

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(
    figsize=(14, 5)
)

plt.plot(
    clean_data["timestamp"],
    clean_data["generator_power"]
)

plt.xlabel(
    "Time"
)

plt.ylabel(
    "Generator Power"
)

plt.title(
    "Generator Power Over Time"
)

plt.tight_layout()

plt.show()

## 16. Define Operating Conditions

The generating unit operates close to its nominal speed during normal operation, while generator power varies according to load demand.

Based on the observed operating behaviour, records are considered operational when unit speed is at least 90 and generator power is greater than zero.

This criterion removes shutdown and non operating periods without excluding valid lower load operating conditions.

In [ ]:
operating_condition_summary = pd.DataFrame({
    "Condition": [
        "Speed >= 90 and Power > 0",
        "Speed < 90 or Power <= 0",
        "Speed >= 90 and Power <= 0",
        "Speed < 90 and Power > 0"
    ],

    "Count": [
        (
            (clean_data["unit_speed"] >= 90)
            & (clean_data["generator_power"] > 0)
        ).sum(),

        (
            (clean_data["unit_speed"] < 90)
            | (clean_data["generator_power"] <= 0)
        ).sum(),

        (
            (clean_data["unit_speed"] >= 90)
            & (clean_data["generator_power"] <= 0)
        ).sum(),

        (
            (clean_data["unit_speed"] < 90)
            & (clean_data["generator_power"] > 0)
        ).sum()
    ]
})

operating_condition_summary

In [ ]:
operating_data = clean_data[
    (clean_data["unit_speed"] >= 90)
    & (clean_data["generator_power"] > 0)
].copy()

operating_data = (
    operating_data
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print(
    "Total cleaned observations:",
    len(clean_data)
)

print(
    "Operating observations:",
    len(operating_data)
)

print(
    "Removed non operating observations:",
    len(clean_data) - len(operating_data)
)

print(
    "Operating data percentage:",
    f"{len(operating_data) / len(clean_data) * 100:.2f}%"
)

In [ ]:
operating_data[
    [
        "generator_power",
        "unit_speed",
        "tgb_temperature",
        "tgb_vibration",
        "ugb_temperature",
        "ugb_vibration"
    ]
].describe()

## 17. Identify Continuous Operating Segments

Removing shutdown periods creates gaps in the time series. Continuous operating segments are therefore identified so that future modelling sequences do not cross shutdown periods or gaps between monthly datasets.

In [ ]:
operating_data["time_gap"] = (
    operating_data["timestamp"]
    .diff()
)

In [ ]:
operating_data["new_segment"] = (
    operating_data["time_gap"]
    != pd.Timedelta(minutes=10)
)

In [ ]:
operating_data.loc[
    0,
    "new_segment"
] = True

In [ ]:
operating_data["segment_id"] = (
    operating_data["new_segment"]
    .cumsum()
)

In [ ]:
print(
    "Number of continuous operating segments:",
    operating_data["segment_id"].nunique()
)

In [ ]:
segment_summary = (
    operating_data
    .groupby("segment_id")
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        observations=("timestamp", "size")
    )
    .reset_index()
)

segment_summary["duration_hours"] = (
    segment_summary["observations"]
    * 10
    / 60
)

segment_summary

In [ ]:
segment_summary[
    [
        "segment_id",
        "observations",
        "duration_hours"
    ]
].sort_values(
    "observations",
    ascending=False
)

In [ ]:
print(
    "Segments with at least 37 observations:",
    (
        segment_summary["observations"] >= 37
    ).sum()
)

## 18. Vibration Data Quality Inspection

The vibration measurements of the Turbine Guide Bearing and Upper Guide Bearing are inspected for zero values, extreme values, and unusual observations before model preparation.

Potential extreme measurements are investigated rather than automatically removed to avoid discarding meaningful component behaviour.

In [ ]:
vibration_quality = pd.DataFrame({
    "Measurement": [
        "TGB vibration = 0",
        "TGB vibration > 10",
        "TGB vibration > 15",
        "UGB vibration = 0",
        "UGB vibration > 15"
    ],

    "Count": [
        (operating_data["tgb_vibration"] == 0).sum(),
        (operating_data["tgb_vibration"] > 10).sum(),
        (operating_data["tgb_vibration"] > 15).sum(),
        (operating_data["ugb_vibration"] == 0).sum(),
        (operating_data["ugb_vibration"] > 15).sum()
    ]
})

vibration_quality

In [ ]:
vibration_percentiles = operating_data[
    [
        "tgb_vibration",
        "ugb_vibration"
    ]
].quantile(
    [
        0.00,
        0.001,
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
        0.999,
        1.00
    ]
)

vibration_percentiles

## 19. Extreme Vibration Observations

The largest vibration measurements are examined together with their timestamps and operating variables to determine whether they represent isolated sensor values or meaningful operating behaviour.

In [ ]:
tgb_highest = operating_data[
    [
        "timestamp",
        "source_file",
        "segment_id",
        "generator_power",
        "unit_speed",
        "tgb_temperature",
        "tgb_vibration",
        "ugb_temperature",
        "ugb_vibration"
    ]
].nlargest(
    20,
    "tgb_vibration"
)

tgb_highest

In [ ]:
ugb_highest = operating_data[
    [
        "timestamp",
        "source_file",
        "segment_id",
        "generator_power",
        "unit_speed",
        "tgb_temperature",
        "tgb_vibration",
        "ugb_temperature",
        "ugb_vibration"
    ]
].nlargest(
    20,
    "ugb_vibration"
)

ugb_highest

In [ ]:
zero_vibration_rows = operating_data[
    (operating_data["tgb_vibration"] == 0)
    |
    (operating_data["ugb_vibration"] == 0)
][
    [
        "timestamp",
        "source_file",
        "segment_id",
        "generator_power",
        "unit_speed",
        "tgb_temperature",
        "tgb_vibration",
        "ugb_temperature",
        "ugb_vibration"
    ]
]

print(
    "Rows containing zero vibration:",
    len(zero_vibration_rows)
)

zero_vibration_rows

In [ ]:
tgb_max_index = (
    operating_data["tgb_vibration"]
    .idxmax()
)

print(
    "Maximum TGB vibration row index:",
    tgb_max_index
)

In [ ]:
start_index = max(
    0,
    tgb_max_index - 10
)

end_index = min(
    len(operating_data) - 1,
    tgb_max_index + 10
)

tgb_max_context = operating_data.loc[
    start_index:end_index,
    [
        "timestamp",
        "segment_id",
        "generator_power",
        "unit_speed",
        "tgb_temperature",
        "tgb_vibration",
        "ugb_temperature",
        "ugb_vibration"
    ]
]

tgb_max_context

In [ ]:
plt.figure(
    figsize=(14, 5)
)

plt.plot(
    operating_data["timestamp"],
    operating_data["tgb_vibration"]
)

plt.xlabel("Time")
plt.ylabel("TGB Vibration")

plt.title(
    "Turbine Guide Bearing Vibration During Operating Conditions"
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(14, 5)
)

plt.plot(
    operating_data["timestamp"],
    operating_data["ugb_vibration"]
)

plt.xlabel("Time")
plt.ylabel("UGB Vibration")

plt.title(
    "Upper Guide Bearing Vibration During Operating Conditions"
)

plt.tight_layout()
plt.show()

In [ ]:
correlation = operating_data[
    [
        "tgb_vibration",
        "ugb_vibration"
    ]
].corr()

correlation

In [ ]:
tgb_ugb_correlation = correlation.loc[
    "tgb_vibration",
    "ugb_vibration"
]

print(
    f"TGB and UGB vibration correlation: "
    f"{tgb_ugb_correlation:.4f}"
)

## 20. Inspect Zero Vibration Measurements

Zero vibration measurements are examined to determine whether they occur simultaneously for the Turbine Guide Bearing and Upper Guide Bearing while the generating unit is operating.

In [ ]:
both_zero = (
    (operating_data["tgb_vibration"] == 0)
    &
    (operating_data["ugb_vibration"] == 0)
).sum()

tgb_zero_only = (
    (operating_data["tgb_vibration"] == 0)
    &
    (operating_data["ugb_vibration"] != 0)
).sum()

ugb_zero_only = (
    (operating_data["ugb_vibration"] == 0)
    &
    (operating_data["tgb_vibration"] != 0)
).sum()

print("Both TGB and UGB = 0:", both_zero)
print("TGB only = 0:", tgb_zero_only)
print("UGB only = 0:", ugb_zero_only)

In [ ]:
zero_rows = operating_data[
    (operating_data["tgb_vibration"] == 0)
    &
    (operating_data["ugb_vibration"] == 0)
].copy()

print("First zero timestamp:")
print(zero_rows["timestamp"].min())

print("\nLast zero timestamp:")
print(zero_rows["timestamp"].max())

print("\nZero rows by source file:")
print(
    zero_rows.groupby(
        "source_file"
    ).size()
)

In [ ]:
zero_rows[
    [
        "timestamp",
        "source_file",
        "generator_power",
        "unit_speed",
        "tgb_temperature",
        "tgb_vibration",
        "ugb_temperature",
        "ugb_vibration"
    ]
].head(20)

In [ ]:
zero_rows = (
    zero_rows
    .sort_values("timestamp")
    .reset_index(drop=True)
)

zero_rows["time_gap"] = (
    zero_rows["timestamp"]
    .diff()
)

zero_rows["new_zero_period"] = (
    zero_rows["time_gap"]
    != pd.Timedelta(minutes=10)
)

zero_rows.loc[
    0,
    "new_zero_period"
] = True

zero_rows["zero_period_id"] = (
    zero_rows["new_zero_period"]
    .cumsum()
)

In [ ]:
zero_period_summary = (
    zero_rows
    .groupby("zero_period_id")
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        observations=("timestamp", "size")
    )
    .reset_index()
)

zero_period_summary

## 21. Remove Invalid Zero Vibration Period

A continuous period was identified where both TGB and UGB vibration measurements were simultaneously zero while the generating unit remained in operation.

Because both connected bearing sensors reported zero for the same extended period, these records are treated as unavailable vibration measurements rather than valid operating behaviour and are removed from the modelling dataset.

In [ ]:
model_data = operating_data[
    ~(
        (operating_data["tgb_vibration"] == 0)
        &
        (operating_data["ugb_vibration"] == 0)
    )
].copy()

model_data = (
    model_data
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print(
    "Operating observations before removal:",
    len(operating_data)
)

print(
    "Observations after removing zero period:",
    len(model_data)
)

print(
    "Rows removed:",
    len(operating_data) - len(model_data)
)

In [ ]:
print(
    "Remaining TGB zeros:",
    (model_data["tgb_vibration"] == 0).sum()
)

print(
    "Remaining UGB zeros:",
    (model_data["ugb_vibration"] == 0).sum()
)

## 22. Reidentify Continuous Operating Segments

The continuous operating segments are recalculated after removing the invalid vibration period so that model sequences cannot cross missing data intervals.

In [ ]:
model_data["time_gap"] = (
    model_data["timestamp"]
    .diff()
)

model_data["new_segment"] = (
    model_data["time_gap"]
    != pd.Timedelta(minutes=10)
)

model_data.loc[
    0,
    "new_segment"
] = True

model_data["segment_id"] = (
    model_data["new_segment"]
    .cumsum()
)

In [ ]:
print(
    "Number of continuous segments after cleaning:",
    model_data["segment_id"].nunique()
)

In [ ]:
segment_summary_clean = (
    model_data
    .groupby("segment_id")
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        observations=("timestamp", "size")
    )
    .reset_index()
)

segment_summary_clean["duration_hours"] = (
    segment_summary_clean["observations"]
    * 10
    / 60
)

segment_summary_clean

In [ ]:
print(
    "Segments with at least 37 observations:",
    (
        segment_summary_clean["observations"] >= 37
    ).sum()
)

## 23. Inspect Segment Start Behaviour

The beginning of each continuous operating segment is inspected to identify possible startup or restart transients before defining a consistent stabilization rule.

In [ ]:
segment_start_rows = (
    model_data
    .groupby("segment_id")
    .head(6)
)

segment_start_rows[
    [
        "segment_id",
        "timestamp",
        "generator_power",
        "unit_speed",
        "tgb_temperature",
        "tgb_vibration",
        "ugb_temperature",
        "ugb_vibration"
    ]
]

In [ ]:
first_row_each_segment = (
    model_data
    .groupby("segment_id")
    .first()
    .reset_index()
)

first_row_each_segment[
    [
        "segment_id",
        "timestamp",
        "generator_power",
        "unit_speed",
        "tgb_temperature",
        "tgb_vibration",
        "ugb_temperature",
        "ugb_vibration"
    ]
]

## 24. Inspect Restart Transients

Segments with unusually high vibration at their first observation are inspected in more detail to determine how quickly the bearing measurements return to their typical operating range.

In [ ]:
segments_to_check = model_data[
    model_data["segment_id"].isin(
        [2, 11]
    )
].copy()

segments_to_check = (
    segments_to_check
    .groupby("segment_id")
    .head(12)
)

segments_to_check[
    [
        "segment_id",
        "timestamp",
        "generator_power",
        "unit_speed",
        "tgb_temperature",
        "tgb_vibration",
        "ugb_temperature",
        "ugb_vibration"
    ]
]

In [ ]:
model_data["position_in_segment"] = (
    model_data
    .groupby("segment_id")
    .cumcount()
)

In [ ]:
restart_check = model_data[
    model_data["segment_id"].isin(
        [2, 11]
    )
].copy()

restart_check = restart_check[
    restart_check["position_in_segment"] < 12
]

restart_check[
    [
        "segment_id",
        "position_in_segment",
        "timestamp",
        "generator_power",
        "unit_speed",
        "tgb_temperature",
        "tgb_vibration",
        "ugb_temperature",
        "ugb_vibration"
    ]
]

In [ ]:
segment_2 = model_data[
    model_data["segment_id"] == 2
].head(12)

plt.figure(
    figsize=(10, 4)
)

plt.plot(
    segment_2["timestamp"],
    segment_2["tgb_vibration"],
    marker="o",
    label="TGB Vibration"
)

plt.plot(
    segment_2["timestamp"],
    segment_2["ugb_vibration"],
    marker="o",
    label="UGB Vibration"
)

plt.xlabel("Time")
plt.ylabel("Vibration")

plt.title(
    "First Two Hours of Segment 2"
)

plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
segment_11 = model_data[
    model_data["segment_id"] == 11
].head(12)

plt.figure(
    figsize=(10, 4)
)

plt.plot(
    segment_11["timestamp"],
    segment_11["tgb_vibration"],
    marker="o",
    label="TGB Vibration"
)

plt.plot(
    segment_11["timestamp"],
    segment_11["ugb_vibration"],
    marker="o",
    label="UGB Vibration"
)

plt.xlabel("Time")
plt.ylabel("Vibration")

plt.title(
    "First Two Hours of Segment 11"
)

plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 25. Remove Initial Segment Transients

Inspection of the operating segment starts showed that unusually high TGB vibration values occurred only at the first observation after a data gap or restart.

The vibration measurements returned to their normal range at the following 10 minute observation. Therefore, the first observation of every continuous operating segment is removed to avoid using possible restart or synchronization transients in the modelling dataset.

In [ ]:
stable_data = model_data[
    model_data["position_in_segment"] > 0
].copy()

stable_data = (
    stable_data
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print(
    "Observations before transient removal:",
    len(model_data)
)

print(
    "Observations after transient removal:",
    len(stable_data)
)

print(
    "Rows removed:",
    len(model_data) - len(stable_data)
)

In [ ]:
print(
    "Maximum TGB vibration:",
    stable_data["tgb_vibration"].max()
)

print(
    "Maximum UGB vibration:",
    stable_data["ugb_vibration"].max()
)

print(
    "Minimum TGB vibration:",
    stable_data["tgb_vibration"].min()
)

print(
    "Minimum UGB vibration:",
    stable_data["ugb_vibration"].min()
)

In [ ]:
stable_data[
    [
        "tgb_vibration",
        "ugb_vibration"
    ]
].describe()

In [ ]:
stable_data[
    [
        "timestamp",
        "source_file",
        "segment_id",
        "generator_power",
        "unit_speed",
        "tgb_temperature",
        "tgb_vibration",
        "ugb_temperature",
        "ugb_vibration"
    ]
].nlargest(
    15,
    "tgb_vibration"
)

In [ ]:
stable_data["position_in_segment"] = (
    stable_data
    .groupby("segment_id")
    .cumcount()
)

## 26. Final Data Quality Check

The cleaned operating dataset is reviewed after removal of incomplete records, non operating periods, invalid zero vibration measurements, and initial segment transients.

In [ ]:
print(
    "Final observations:",
    len(stable_data)
)

print(
    "\nMissing values:"
)

print(
    stable_data[
        [
            "timestamp",
            "generator_power",
            "unit_speed",
            "tgb_temperature",
            "tgb_vibration",
            "ugb_temperature",
            "ugb_vibration"
        ]
    ].isna().sum()
)

print(
    "\nTGB zeros:",
    (stable_data["tgb_vibration"] == 0).sum()
)

print(
    "UGB zeros:",
    (stable_data["ugb_vibration"] == 0).sum()
)

In [ ]:
stable_data.groupby(
    "source_file"
).size()

## 27. Save Cleaned Master Dataset

The final cleaned operating dataset is saved before model specific preparation.

The dataset contains valid operating measurements for both the Turbine Guide Bearing and Upper Guide Bearing after removal of incomplete records, non operating periods, invalid simultaneous zero vibration measurements, and initial segment transients.

In [ ]:
cleaned_file = (
    PREPARED_DIR /
    "hydrotwin_cleaned_data.csv"
)

stable_data.to_csv(
    cleaned_file,
    index=False
)

print(
    "Saved cleaned dataset:"
)

print(cleaned_file)

In [ ]:
print(
    "File exists:",
    cleaned_file.exists()
)

## 28. Chronological Dataset Split

The cleaned dataset is divided chronologically to prevent information leakage.

June, July, and August are used for model training. September is used for validation and model development. November is reserved as the final unseen test dataset.

October is excluded because only 33 valid operating observations remain after data quality filtering, which is insufficient for a 36 observation input window and one step prediction target.

In [ ]:
train_data = stable_data[
    stable_data["timestamp"].dt.month.isin(
        [6, 7, 8]
    )
].copy()

val_data = stable_data[
    stable_data["timestamp"].dt.month == 9
].copy()

test_data = stable_data[
    stable_data["timestamp"].dt.month == 11
].copy()

october_data = stable_data[
    stable_data["timestamp"].dt.month == 10
].copy()

In [ ]:
print(
    "Training observations:",
    len(train_data)
)

print(
    "Validation observations:",
    len(val_data)
)

print(
    "October excluded observations:",
    len(october_data)
)

print(
    "Test observations:",
    len(test_data)
)

In [ ]:
split_summary = pd.DataFrame({
    "Split": [
        "Training",
        "Validation",
        "Excluded October",
        "Final Test"
    ],

    "Observations": [
        len(train_data),
        len(val_data),
        len(october_data),
        len(test_data)
    ],

    "Start": [
        train_data["timestamp"].min(),
        val_data["timestamp"].min(),
        october_data["timestamp"].min(),
        test_data["timestamp"].min()
    ],

    "End": [
        train_data["timestamp"].max(),
        val_data["timestamp"].max(),
        october_data["timestamp"].max(),
        test_data["timestamp"].max()
    ]
})

split_summary

## 29. Define Component Input Variables

Both guide bearing models use the same four types of input information to maintain a consistent modelling framework.

For TGB, the inputs are TGB vibration, TGB temperature, unit speed, and generator power.

For UGB, the inputs are UGB vibration, UGB temperature, unit speed, and generator power.

The prediction target for each component is its vibration value at the next 10 minute interval.

In [ ]:
TGB_FEATURES = [
    "tgb_vibration",
    "tgb_temperature",
    "unit_speed",
    "generator_power"
]

UGB_FEATURES = [
    "ugb_vibration",
    "ugb_temperature",
    "unit_speed",
    "generator_power"
]

TGB_TARGET = "tgb_vibration"

UGB_TARGET = "ugb_vibration"

SEQUENCE_LENGTH = 36

In [ ]:
print(
    "History window:",
    SEQUENCE_LENGTH,
    "observations"
)

print(
    "History duration:",
    SEQUENCE_LENGTH * 10 / 60,
    "hours"
)

print(
    "Forecast horizon: 10 minutes"
)

## 30. Create Continuous Time Series Sequences

Each modelling sample contains the previous 36 continuous observations, representing six hours of historical measurements.

The target is the component vibration at the following 10 minute interval.

Sequences are created separately within each continuous operating segment so that no sample crosses a shutdown, missing data period, or other time gap.

In [ ]:
def create_sequences(
    data,
    feature_columns,
    target_column,
    sequence_length=36
):

    X = []
    y = []
    timestamps = []
    segment_ids = []

    for segment_id, segment in data.groupby(
        "segment_id"
    ):

        segment = (
            segment
            .sort_values("timestamp")
            .reset_index(drop=True)
        )

        if len(segment) <= sequence_length:
            continue

        feature_values = segment[
            feature_columns
        ].to_numpy()

        target_values = segment[
            target_column
        ].to_numpy()

        time_values = segment[
            "timestamp"
        ].to_numpy()

        for i in range(
            sequence_length,
            len(segment)
        ):

            X.append(
                feature_values[
                    i - sequence_length:i
                ]
            )

            y.append(
                target_values[i]
            )

            timestamps.append(
                time_values[i]
            )

            segment_ids.append(
                segment_id
            )

    return (
        np.array(X),
        np.array(y),
        np.array(timestamps),
        np.array(segment_ids)
    )

In [ ]:
X_train_tgb, y_train_tgb, train_times_tgb, train_segments_tgb = (
    create_sequences(
        train_data,
        TGB_FEATURES,
        TGB_TARGET,
        SEQUENCE_LENGTH
    )
)

X_val_tgb, y_val_tgb, val_times_tgb, val_segments_tgb = (
    create_sequences(
        val_data,
        TGB_FEATURES,
        TGB_TARGET,
        SEQUENCE_LENGTH
    )
)

X_test_tgb, y_test_tgb, test_times_tgb, test_segments_tgb = (
    create_sequences(
        test_data,
        TGB_FEATURES,
        TGB_TARGET,
        SEQUENCE_LENGTH
    )
)

In [ ]:
print("TGB")

print(
    "X_train:",
    X_train_tgb.shape
)

print(
    "y_train:",
    y_train_tgb.shape
)

print(
    "X_val:",
    X_val_tgb.shape
)

print(
    "y_val:",
    y_val_tgb.shape
)

print(
    "X_test:",
    X_test_tgb.shape
)

print(
    "y_test:",
    y_test_tgb.shape
)

In [ ]:
X_train_ugb, y_train_ugb, train_times_ugb, train_segments_ugb = (
    create_sequences(
        train_data,
        UGB_FEATURES,
        UGB_TARGET,
        SEQUENCE_LENGTH
    )
)

X_val_ugb, y_val_ugb, val_times_ugb, val_segments_ugb = (
    create_sequences(
        val_data,
        UGB_FEATURES,
        UGB_TARGET,
        SEQUENCE_LENGTH
    )
)

X_test_ugb, y_test_ugb, test_times_ugb, test_segments_ugb = (
    create_sequences(
        test_data,
        UGB_FEATURES,
        UGB_TARGET,
        SEQUENCE_LENGTH
    )
)

In [ ]:
print("UGB")

print(
    "X_train:",
    X_train_ugb.shape
)

print(
    "y_train:",
    y_train_ugb.shape
)

print(
    "X_val:",
    X_val_ugb.shape
)

print(
    "y_val:",
    y_val_ugb.shape
)

print(
    "X_test:",
    X_test_ugb.shape
)

print(
    "y_test:",
    y_test_ugb.shape
)

In [ ]:
print(
    "Training samples aligned:",
    np.array_equal(
        train_times_tgb,
        train_times_ugb
    )
)

print(
    "Validation samples aligned:",
    np.array_equal(
        val_times_tgb,
        val_times_ugb
    )
)

print(
    "Test samples aligned:",
    np.array_equal(
        test_times_tgb,
        test_times_ugb
    )
)

## 31. Verify Sequence and Target Alignment

A sample sequence is inspected to confirm that the 36 historical observations are used to predict the component vibration at the following 10 minute interval.

In [ ]:
sample_index = 0

print(
    "TGB input shape:",
    X_train_tgb[
        sample_index
    ].shape
)

print(
    "TGB target:",
    y_train_tgb[
        sample_index
    ]
)

print(
    "Target timestamp:",
    train_times_tgb[
        sample_index
    ]
)

print(
    "Segment:",
    train_segments_tgb[
        sample_index
    ]
)

In [ ]:
first_tgb_sequence = pd.DataFrame(
    X_train_tgb[0],
    columns=TGB_FEATURES
)

first_tgb_sequence

In [ ]:
print(
    "Number of historical observations:",
    len(first_tgb_sequence)
)

print(
    "Target vibration:",
    y_train_tgb[0]
)

print(
    "Target timestamp:",
    train_times_tgb[0]
)

In [ ]:
first_segment = (
    train_data[
        train_data["segment_id"]
        == train_segments_tgb[0]
    ]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

first_segment[
    [
        "timestamp",
        "tgb_vibration",
        "ugb_vibration"
    ]
].head(40)

In [ ]:
print(
    "Expected target timestamp:",
    first_segment.loc[
        36,
        "timestamp"
    ]
)

print(
    "Saved target timestamp:",
    train_times_tgb[0]
)

print(
    "Expected TGB target:",
    first_segment.loc[
        36,
        "tgb_vibration"
    ]
)

print(
    "Saved TGB target:",
    y_train_tgb[0]
)

In [ ]:
print(
    "Expected UGB target:",
    first_segment.loc[
        36,
        "ugb_vibration"
    ]
)

print(
    "Saved UGB target:",
    y_train_ugb[0]
)

## 32. Final Sequence Integrity Check

The prepared sequence arrays are checked for missing or non finite values before they are saved for model development.

In [ ]:
arrays_to_check = {
    "X_train_tgb": X_train_tgb,
    "y_train_tgb": y_train_tgb,
    "X_val_tgb": X_val_tgb,
    "y_val_tgb": y_val_tgb,
    "X_test_tgb": X_test_tgb,
    "y_test_tgb": y_test_tgb,

    "X_train_ugb": X_train_ugb,
    "y_train_ugb": y_train_ugb,
    "X_val_ugb": X_val_ugb,
    "y_val_ugb": y_val_ugb,
    "X_test_ugb": X_test_ugb,
    "y_test_ugb": y_test_ugb
}

for name, array in arrays_to_check.items():

    print(
        name,
        "finite:",
        np.isfinite(array).all()
    )

## 33. Save Prepared Sequences

The unscaled sequence arrays are saved so that Linear Regression, LSTM, and XGBoost can use exactly the same training, validation, and test samples.

Any model specific scaling or reshaping will be performed later using only the training data.

In [ ]:
TGB_DIR = PREPARED_DIR / "tgb"
UGB_DIR = PREPARED_DIR / "ugb"

TGB_DIR.mkdir(
    parents=True,
    exist_ok=True
)

UGB_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
np.save(
    TGB_DIR / "X_train.npy",
    X_train_tgb
)

np.save(
    TGB_DIR / "y_train.npy",
    y_train_tgb
)

np.save(
    TGB_DIR / "X_val.npy",
    X_val_tgb
)

np.save(
    TGB_DIR / "y_val.npy",
    y_val_tgb
)

np.save(
    TGB_DIR / "X_test.npy",
    X_test_tgb
)

np.save(
    TGB_DIR / "y_test.npy",
    y_test_tgb
)

np.save(
    TGB_DIR / "train_times.npy",
    train_times_tgb
)

np.save(
    TGB_DIR / "val_times.npy",
    val_times_tgb
)

np.save(
    TGB_DIR / "test_times.npy",
    test_times_tgb
)

In [ ]:
np.save(
    UGB_DIR / "X_train.npy",
    X_train_ugb
)

np.save(
    UGB_DIR / "y_train.npy",
    y_train_ugb
)

np.save(
    UGB_DIR / "X_val.npy",
    X_val_ugb
)

np.save(
    UGB_DIR / "y_val.npy",
    y_val_ugb
)

np.save(
    UGB_DIR / "X_test.npy",
    X_test_ugb
)

np.save(
    UGB_DIR / "y_test.npy",
    y_test_ugb
)

np.save(
    UGB_DIR / "train_times.npy",
    train_times_ugb
)

np.save(
    UGB_DIR / "val_times.npy",
    val_times_ugb
)

np.save(
    UGB_DIR / "test_times.npy",
    test_times_ugb
)

In [ ]:
np.save(
    TGB_DIR / "train_segments.npy",
    train_segments_tgb
)

np.save(
    TGB_DIR / "val_segments.npy",
    val_segments_tgb
)

np.save(
    TGB_DIR / "test_segments.npy",
    test_segments_tgb
)

np.save(
    UGB_DIR / "train_segments.npy",
    train_segments_ugb
)

np.save(
    UGB_DIR / "val_segments.npy",
    val_segments_ugb
)

np.save(
    UGB_DIR / "test_segments.npy",
    test_segments_ugb
)

In [ ]:
import json

In [ ]:
metadata = {
    "project": "HydroTwin",

    "components": [
        "Turbine Guide Bearing",
        "Upper Guide Bearing"
    ],

    "sampling_interval_minutes": 10,

    "sequence_length": 36,

    "history_hours": 6,

    "forecast_horizon_minutes": 10,

    "tgb_features": TGB_FEATURES,

    "ugb_features": UGB_FEATURES,

    "tgb_target": TGB_TARGET,

    "ugb_target": UGB_TARGET,

    "training_months": [
        "June",
        "July",
        "August"
    ],

    "validation_month": "September",

    "excluded_month": "October",

    "test_month": "November",

    "train_samples": int(
        len(X_train_tgb)
    ),

    "validation_samples": int(
        len(X_val_tgb)
    ),

    "test_samples": int(
        len(X_test_tgb)
    )
}

In [ ]:
metadata_path = (
    PREPARED_DIR /
    "dataset_metadata.json"
)

with open(
    metadata_path,
    "w"
) as file:

    json.dump(
        metadata,
        file,
        indent=4
    )

print(
    "Metadata saved to:"
)

print(metadata_path)

In [ ]:
print("TGB files:")

for file in sorted(
    TGB_DIR.iterdir()
):
    print(file.name)

In [ ]:
print("UGB files:")

for file in sorted(
    UGB_DIR.iterdir()
):
    print(file.name)